In [20]:
import os
import random
from asgiref.sync import sync_to_async

from typing import cast

import logging
import tiktoken
import openai

# from django.db.models import QuerySet

from literev.models import Cluster, Document
from literev.libs.nlp import build_prompt, call_chatgpt
from asgiref.sync import sync_to_async

logger = logging.getLogger(__name__)


In [21]:
API_KEY = "sk-proj-BW4s0WknbzkGoxpzaDHZT3BlbkFJnH2sD1NS1t7extu5sGVc"
# Maximum tokens supported by the model
GPT_MODEL_MAX_TOKENS = 4096
# Number of tokens reserved for GPT's response
MAX_TOKENS_RESPONSE = 250
# Safety margin to avoid hitting the limit exactly
SAFETY_MARGIN = 50
# Adjusted token limit for the input prompt
TOKEN_LIMIT = GPT_MODEL_MAX_TOKENS - MAX_TOKENS_RESPONSE - SAFETY_MARGIN


### Saved Prompts and Summary results

In [22]:
# Build and print a prompt for the given cluster ID.
@sync_to_async
def ret_build_prompt(cluster_id):
    cluster = (Cluster.objects.get)(id=cluster_id)
    prompt = build_prompt(cluster)

    print(f"Saved prompt for cluster_id: {cluster_id}", "\n\n", prompt)
    print("\n", 50 * "---", "\n")
    print(cluster.topic)    
    print("\n", 50 * "---", "\n")
    print(f"Summary Cluster for cluster_id: {cluster_id}", "\n\n", cluster.summary)


In [23]:
for i in [278,279]:
    print(50 * "===")
    await ret_build_prompt(i) # Clusters IDs 279, 278...

Saved prompt for cluster_id: 278 

 Provide a clear and concise general description based on the following most important keywords: détention, prestation, mme, ocpm, entretien, contribuable, hospice, fiscal, liasi, reconsidération from a cluster containing law cases from the canton of Geneva in Switzerland. The summary must represent all law cases collectively and not be based only on a single one. In addition to the most important keywords, use the following additional context coming from the law cases contained in this cluster:

Document number: 1
Descriptor: 
Keywords Sample: ocpm hospic hospice verser prestation montant total chf ocpm délivrer intégration réussir intégration leu recherche emploi diplôme relativiser puisqu arriver âge garder attache pays résider mère frère.

Document number: 2
Descriptor: 
Keywords Sample: renseignement utile renoncer subir préjudice exercice garde fille cumulatif défaut avantage indu occurrence versement rétroactif prestation frais temporaire enfer

### `chatgpt_function`

In [24]:
def custom_call_chatgpt(
    gpt_model,
    prompt: str,
    temperature: float = 0.4,
    max_tokens_response=250,
    api_key: str = API_KEY) -> str:
    """
    """
    openai_client = openai.OpenAI(api_key=API_KEY)
    
    messages = [{"role": "user", "content": prompt}]

    try:
        logger.info("Sending request to OpenAI ChatGPT API")
        response = openai_client.chat.completions.create(
            model=gpt_model,
            messages=messages,  # type: ignore
            max_tokens=max_tokens_response,
            temperature=temperature,  # this sets the creativity of the response
        )
        print(f"Default GPT_MODEL: {gpt_model}")
        print(f"Temperature: {temperature}")
        # print(f"Prompt: {messages}")

    except openai.RateLimitError as e:
        if settings.DEBUG:
            logger.debug(f"RateLimitError in dev mode: {e}")
            return "RateLimit Error devmode dummy text"
        else:
            # if not in dev mode, log error and return empty string
            logger.error(f"OpenAI API RateLimitError: {e}")
            return ""
    except openai.OpenAIError as e:
        # all other OpenAI api errors will fall here and return an empty string
        logger.error(f"OpenAI API Error:{e}")
        return ""

    return cast(str, response.choices[0].message.content)


### `build_prompt`

In [25]:
def custom_build_prompt(
    cluster: Cluster,
    gpt_model,
    prompt_template: str, 
    prompt_constraints: str,
    random_k: int = 10
) -> str:
    """
    """

    # this template is used for each document
    document_prompt_template = (
        "Document number: {document_number}\n"
        "Descriptor: {document_descriptor}\n"
        "Keywords Sample: {document_abstract}.\n\n"
    )

    # inject topic keyword into prompt
    prompt = prompt_template.format(keywords=cluster.topic)

    # Attempt to use the tokenizer for the specified model, fallback if unavailable
    enc = tiktoken.encoding_for_model(gpt_model)

    # get all documents related to cluster(topic)
    documents = Document.objects.filter(clusterelement__cluster=cluster)

    # calculate initial number of tokens
    tokens_count = len(enc.encode(prompt)) + len(
        enc.encode(prompt_constraints)
    )

    # loop through all documents
    for document_number, document in enumerate(documents, start=1):
        # get document title
        document_descriptor = document.descriptors

        # get document abstract sentences by splitting it on "."
        abstract_words = document.preprocessed_document.split()
        
        # print("abstract_words:", abstract_words)

        document_abstract = (
            
            " ".join(
                random.choices(
                    population=abstract_words, k=random_k
                )  # TODO: take random adjacent words
            )
            if len(abstract_words) > random_k
            else document.preprocessed_document
        )

        # fill the document prompt template with current loop's document data
        document_prompt = document_prompt_template.format(
            document_number=document_number,
            document_descriptor=document_descriptor,
            document_abstract=document_abstract,
        )

        # count number of tokens in the document prompt
        document_tokens = len(enc.encode(document_prompt))

        # update tokens count for the next loop
        tokens_count = tokens_count + document_tokens

        # check if the token limit was reached
        if tokens_count > TOKEN_LIMIT:
            # break the loop without appending the document to the prompt

            # print(
            #     f"TOTAL_TOKENS = tokens_count: {tokens_count} + document_tokens: {document_tokens}"
            #     )              
            print("Skiping...", tokens_count, TOKEN_LIMIT)

        # limit not reached, append the document prompt to the 'main' prompt
        prompt = prompt + document_prompt

    print(
        f"TOTAL_TOKENS = tokens_count: {tokens_count} + document_tokens: {document_tokens}"
        )
    
    # add constraints at the end of the prompt after the loop
    # gpt performs better if we pass the constraints at the end
    prompt = prompt + prompt_constraints
    return prompt

### Generate a topic description based on the cluster data

In [26]:
@sync_to_async
def custom_nlp_topic_description(
    cluster: Cluster | int, 
    gpt_model,
    prompt_template,
    prompt_constraints,
    random_k,
    api_key: str = API_KEY
) -> str:
    """"""
    if isinstance(cluster, int):
        cluster = Cluster.objects.get(id=cluster)


    prompt = custom_build_prompt(
        cluster, 
        gpt_model,
        prompt_template, 
        prompt_constraints,
        random_k
    )
    print(prompt)
    return custom_call_chatgpt(
        gpt_model,        
        prompt=prompt, 
        temperature=0, 
        max_tokens_response=250, 
        api_key=api_key
    )


### Input using the old prompt template and `gpt-3.5-turbo`

In [27]:

cluster_id =  279
gpt_model = "gpt-3.5-turbo"
random_k=10

prompt_template = (
    "Provide a succint general description"
    "based on the following most important keywords: {keywords} selected"
    "from a cluster containing case law from the canton of Geneva in Switzerland."
    "The following descriptors for each case law provide additional context:\n\n"
)

# constraints can be fine-tuned to prevent unwanted words or sentences
# this is to prevent unwanted words or sentences
prompt_constraints = (
    "Ensure the response maintains a narrative voice suitable for general "
    "description while minimizing the use of pronouns. Avoid unnecessary "
    "introductions and redundancies. Avoid using quotes and backticks "
    "Avoid the use of the following keywords and expressions: keyword, "
    "keywords, topic, case, we propose, "
    "proposing, we, this report. Summarize it in two sentences in french language."
)


In [28]:

await  custom_nlp_topic_description(
    cluster_id, 
    gpt_model,
    prompt_template,
    prompt_constraints,
    random_k=random_k,
    api_key=API_KEY)


TOTAL_TOKENS = tokens_count: 3323 + document_tokens: 92
Provide a succint general descriptionbased on the following most important keywords: ocpm, leu, pays, intégration, mme, conjugal, réintégration, mariage, fille, épouse selectedfrom a cluster containing case law from the canton of Geneva in Switzerland.The following descriptors for each case law provide additional context:

Document number: 1
Descriptor: 
Keywords Sample: travailler mariage reconnaître compagne sortie mariage compagne constitutionnel enceint impartir.

Document number: 2
Descriptor: 
Keywords Sample: concubinage parent compagne frontalier_déplacement juger retrouver mineur réadapter ensembl sentimental.

Document number: 3
Descriptor: 
Keywords Sample: fils rendez épouse regard alléguer âg ordinaire mener créancier police.

Document number: 4
Descriptor: 
Keywords Sample: union appartement séparer contredire bancaire salon_massage quasi réputer essentiel montant.

Document number: 5
Descriptor: DROIT DES ÉTRANGERS;

"Les cas de jurisprudence du canton de Genève en Suisse abordent des questions telles que le mariage, la réintégration familiale, l'intégration sociale, et les autorisations de séjour pour les ressortissants étrangers. Les décisions portent sur des sujets tels que les conjoints, les enfants, les conjoints de fait, et les situations de rigueur, mettant en lumière des aspects juridiques et sociaux liés à la vie familiale et conjugale."

### Input using the current prompt template and `gpt-4o-mini`

In [29]:

cluster_id =  278
gpt_model = "gpt-4o-mini"
random_k=27

prompt_template = (
    "Provide a clear and concise general description based on the following "
    "most important keywords: ```{keywords}``` selected "
    "from a cluster containing case law from the canton of Geneva in Switzerland. "
    "The following descriptors for each case law provide additional context:\n\n"
)

prompt_constraints = (
    "Ensure the response maintains a narrative voice suitable for general "
    "description while minimizing the use of pronouns. Avoid unnecessary "
    "introductions and redundancies. Avoid using quotes and backticks. "
    "Avoid the use of the following keywords and expressions: ```keyword, "
    "keywords, topic, case, we propose, proposing, we, this report```. "
    "Do not include names or generate the answer based solely on a single article. "
    "Summarize it in two sentences in French language."
)

In [30]:

await  custom_nlp_topic_description(
    cluster_id, 
    gpt_model,
    prompt_template,
    prompt_constraints,
    random_k=random_k,
    api_key=API_KEY)


Skiping... 3802 3796
Skiping... 3859 3796
Skiping... 3915 3796
Skiping... 3964 3796
Skiping... 4018 3796
Skiping... 4178 3796
Skiping... 4229 3796
Skiping... 4452 3796
Skiping... 4589 3796
Skiping... 4745 3796
Skiping... 4808 3796
Skiping... 4972 3796
Skiping... 5129 3796
Skiping... 5188 3796
Skiping... 5292 3796
Skiping... 5349 3796
Skiping... 5409 3796
TOTAL_TOKENS = tokens_count: 5409 + document_tokens: 60
Provide a clear and concise general description based on the following most important keywords: ```détention, prestation, mme, ocpm, entretien, contribuable, hospice, fiscal, liasi, reconsidération``` selected from a cluster containing case law from the canton of Geneva in Switzerland. The following descriptors for each case law provide additional context:

Document number: 1
Descriptor: 
Keywords Sample: pratique hospic impossible grave pays trouble accès entretien dépressif_récurrer écriture souffrir annuler consultation trouble cappi contestation inférieur entrée leu recommande

"La détention et la prestation d'assistance sont des enjeux centraux dans les affaires traitées par l'OCPM, où des contribuables, souvent en situation de vulnérabilité, cherchent à obtenir des révisions fiscales ou des aides financières. Les demandes de reconsidération, notamment en lien avec des situations d'hospice ou des obligations d'entretien, soulignent la complexité des interactions entre les droits individuels et les exigences administratives."

### Input using the `improved` prompt template and `gpt-4o-mini`

---

### Prompt Optimization

In [31]:
cluster_id =  279
gpt_model = "gpt-4o-mini"
random_k=27


# Base prompt used for generating the topic description
   
prompt_template = (
    "Provide a clear and concise general description based on the following "
    "most important keywords: {keywords} from a cluster containing "
    "law cases from the canton of Geneva in Switzerland. The summary must "
    "represent all law cases collectively and not be based only on a single one. "
    "In addition to the most important keywords, use "
    "the following additional context coming from the law cases contained in this cluster:"
     "\n\n"
)

prompt_constraints = (
    "Ensure the response maintains a narrative voice suitable for a general "
    "description while minimizing the use of pronouns. "
    "Avoid: unnecessary introductions and redundancies, using quotes, backticks, "
    "cluster's keywords and names of individuals, and words such as topic, case, "
    "we propose, proposing, we, this report. Summarize in exactly "
    "two sentences, ensuring no redundancy between sentences. "
    "Write the response in French."
)


In [32]:

def optimized_build_prompt(
    cluster: Cluster,
    gpt_model: str,
    prompt_template: str,
    prompt_constraints: str,
    max_tokens_response: int = 250,
    random_k: int = 10,
    importance_func=None,  # Function to prioritize documents
) -> str:
    """
    Build an optimized prompt for a scientific topic description.

    """
    print("randomk",  random_k)
    # GPT model max tokens (4096 for GPT-4o-mini)
    gpt_model_max_tokens = 128000

    # Calculate token budget
    token_limit = gpt_model_max_tokens - MAX_TOKENS_RESPONSE - 200

    # Initialize tokenizer
    enc = tiktoken.encoding_for_model(gpt_model)


    # Inject keywords into prompt
    prompt = prompt_template.format(keywords=cluster.topic)
    base_tokens = len(enc.encode(prompt)) + len(enc.encode(prompt_constraints))

    # Fetch documents and optionally prioritize
    documents = Document.objects.filter(clusterelement__cluster=cluster)
    if importance_func:
        documents = sorted(documents, key=importance_func, reverse=True)

    # Template for document descriptions
    document_prompt_template = (
        "Document number: {document_number}\n"
        "Descriptor: {document_descriptor}\n"
        "Keywords Sample: {document_abstract}.\n\n"
    )

    # Track remaining tokens
    remaining_tokens = token_limit - base_tokens
    prompt_fragments = []

    # Iterate through documents to build the prompt
    for document_number, document in enumerate(documents, start=1):
        # Extract relevant details
        document_descriptor = document.descriptors
    
        abstract_words = document.preprocessed_document.split()
        rnd_index = random.randrange(len(abstract_words)- random_k)
        # available_k = min(len(abstract_words), random_k, remaining_tokens // 10)

        document_abstract = (
            # get 27 random adjacent words from preprocessed text
            " ".join(
                abstract_words[rnd_index:rnd_index+random_k]
            )
            if len(abstract_words) > random_k
            else document.preprocessed_document
        )
        
        # Build document-specific prompt
        document_prompt = document_prompt_template.format(
            document_number=document_number,
            document_descriptor=document_descriptor,
            document_abstract=document_abstract,
        )

        # Count tokens for the document prompt
        document_tokens = len(enc.encode(document_prompt))

        # Check if the document fits within the remaining token budget
        if document_tokens > remaining_tokens:
            break  # Stop adding documents

        # Add document prompt to the main fragments
        prompt_fragments.append(document_prompt)
        remaining_tokens -= document_tokens

    # Finalize prompt with constraints
    full_prompt = prompt + "".join(prompt_fragments) + prompt_constraints
    return full_prompt

### Generate a topic description based on the cluster data

In [33]:

@sync_to_async
def custom_nlp_topic_description(
    cluster: Cluster | int, 
    gpt_model,
    prompt_template,
    prompt_constraints,
    random_k,
    api_key: str = API_KEY
) -> str:
    """"""
    if isinstance(cluster, int):
        cluster = Cluster.objects.get(id=cluster)


    prompt = optimized_build_prompt(
        cluster, 
        gpt_model,
        prompt_template, 
        prompt_constraints,
        random_k=random_k,
        # importance_func=prioritize_by_keyword_density
    )

    print(prompt)
    
    return custom_call_chatgpt(
        gpt_model,        
        prompt=prompt, 
        temperature=0, 
        max_tokens_response=250, 
        api_key=api_key
    )


In [ ]:

await  custom_nlp_topic_description(
    cluster_id, 
    gpt_model,
    prompt_template,
    prompt_constraints,
    random_k=27,
    api_key=API_KEY)


randomk 27
Provide a clear and concise general description based on the following most important keywords: ocpm, leu, pays, intégration, mme, conjugal, réintégration, mariage, fille, épouse from a cluster containing law cases from the canton of Geneva in Switzerland. The summary must represent all law cases collectively and not be based only on a single one. In addition to the most important keywords, use the following additional context coming from the law cases contained in this cluster:

Document number: 1
Descriptor: 
Keywords Sample: quitter mariage reconnaître civil couple émarger acquérir niveau français inscrire cours français intégration leu impliquer convention sauvegarde homme liberté fondamentale cedh subsidiairement ocpm examen contrat indéterminé salaire.

Document number: 2
Descriptor: 
Keywords Sample: ressortissant retrouver pays matériel affectif obtention baccalauréat court programme maîtrise traduction souhaiter acquérir formation complet activité exercer assistance